# Instruction Tuning with SFT and LoRA

Pretraining produces a model that completes text. SFT produces a model that follows instructions. The gap is not about capability — the knowledge is already there. It is about [*behavior*]{.underline}: shaping the model's output toward structured responses to human questions. This notebook covers the SFT objective, chat templates, the `InstructDataset`, catastrophic forgetting, and LoRA — the [low-rank adaptation]{.mark} technique that fine-tunes a 29.9M-parameter model using under 2% of its parameters.

## The SFT Objective

Pretraining minimizes cross-entropy over [every]{.underline} token in the training corpus:

$$\mathcal{L}_{\text{pretrain}} = -\frac{1}{T} \sum_{t=1}^{T} \log p(x_t \mid x_{<t}).$$

SFT data has structure: each sample is a conversation with a prompt $\mathcal{P}$ (system turn + user turn) and a response $\mathcal{R}$ (assistant turn). We only want the model to learn from the response — penalizing it for not predicting the prompt tokens would reinforce the arbitrary choice of prompt wording rather than the quality of the answer. The SFT objective masks out all prompt positions:

$$\mathcal{L}_{\text{SFT}} = -\frac{1}{|\mathcal{R}|} \sum_{t \in \mathcal{R}} \log p(x_t \mid x_{<t}).$$

In PyTorch, this is implemented via the `ignore_index` argument of `F.cross_entropy`. Positions where `targets == -100` contribute [zero loss and zero gradient]{.mark}:

In [ ]:
loss = F.cross_entropy(
    logits.view(-1, vocab_size),
    targets.view(-1),
    ignore_index=-100,   # standard mask value
)
# Positions where targets == -100 contribute zero loss and zero gradient

## Chat Templates

A **chat template** is a function that converts a list of conversation turns into a single token sequence. The exact format varies by model family; what matters is that it is consistent between training and inference.

We use a simple format for the nano model:

```
<|system|>
You are a helpful assistant.
<|user|>
What is the capital of France?
<|assistant|>
The capital of France is Paris.
<|end|>
```

Each special token (`<|system|>`, `<|user|>`, `<|assistant|>`, `<|end|>`) must be in the tokenizer's vocabulary. For the nano tokenizer trained on TinyShakespeare, we add them as special tokens before fine-tuning.

We implement `format_chat` (which serializes a conversation to a string) and `build_loss_mask` (which marks assistant response positions for loss computation):

In [ ]:
from dataclasses import dataclass
from typing import Literal

Role = Literal['system', 'user', 'assistant']

@dataclass
class Message:
    role:    Role
    content: str

SPECIAL_TOKENS = {
    'system':    '<|system|>',
    'user':      '<|user|>',
    'assistant': '<|assistant|>',
    'end':       '<|end|>',
}

def format_chat(
    messages:       list[Message],
    add_gen_prompt: bool = False,
) -> str:
    """
    Convert a list of messages to a single string using the chat template.

    add_gen_prompt=True appends '<|assistant|>\n' at the end — used during
    inference to prime the model to generate the assistant's response.
    """
    parts = []
    for msg in messages:
        role_token = SPECIAL_TOKENS[msg.role]
        parts.append(f"{role_token}\n{msg.content.strip()}\n{SPECIAL_TOKENS['end']}\n")

    text = ''.join(parts)
    if add_gen_prompt:
        text += f"{SPECIAL_TOKENS['assistant']}\n"
    return text


def build_loss_mask(
    tokens:    list[int],
    tokenizer,
) -> list[int]:
    """
    Build the loss mask for a formatted chat sequence.
    Returns a list of the same length as `tokens`:
    -  token_id  at assistant response positions (loss computed here)
    - -100       at all other positions (prompt, system, user turns — masked)
    """
    mask   = [-100] * len(tokens)          # <1>
    text   = tokenizer.decode(tokens)

    # Find all assistant response spans
    asst_start_tok = tokenizer.encode(SPECIAL_TOKENS['assistant'])[0]  # <2>
    end_tok        = tokenizer.encode(SPECIAL_TOKENS['end'])[0]

    in_response = False
    for i, tok in enumerate(tokens):
        if tok == asst_start_tok:
            in_response = True
            # mask the <|assistant|> token itself — model should not
            # "predict" the role marker, only the content after it
            continue
        if tok == end_tok and in_response:  # <3>
            # Include the <|end|> token in the loss — model must learn to stop
            mask[i] = tok
            in_response = False
            continue
        if in_response:
            mask[i] = tok

    return mask

1. Start with all positions masked; unmask only assistant response spans.
2. Special tokens are single-token strings — we encode once and look them up by token id during the scan.
3. The `<|end|>` token closes the response. We include it in the loss so the model learns to generate the stop signal.

## The SFT Dataset

The `InstructDataset` class loads conversation data from a JSONL file, formats each sample with the chat template, builds the loss mask, and returns `(input_ids, labels)` pairs suitable for the training loop. The key implementation detail is the [label shift]{.mark}: `labels[t]` must equal `tokens[t+1]`, because the model at position $t$ predicts the next token. We build the mask over the full token sequence, then shift both input and labels by one:

In [ ]:
import json
import torch
from torch.utils.data import Dataset
from pathlib import Path

class InstructDataset(Dataset):
    """
    Loads instruction-following data from a JSONL file where each line is:
    {
        "messages": [
            {"role": "system",    "content": "You are a helpful assistant."},
            {"role": "user",      "content": "What is 2 + 2?"},
            {"role": "assistant", "content": "2 + 2 equals 4."}
        ]
    }

    Returns (input_ids, labels) pairs where:
    - input_ids: token IDs of the full conversation
    - labels:    token IDs at assistant positions, -100 elsewhere
    """

    def __init__(
        self,
        data_path:  str,
        tokenizer,
        max_length: int = 512,
        pad_to_max: bool = False,
    ):
        self.tokenizer   = tokenizer
        self.max_length  = max_length
        self.pad_to_max  = pad_to_max
        self.samples     = []

        with open(data_path) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    self.samples.append(json.loads(line))
                except json.JSONDecodeError:
                    continue

        print(f"InstructDataset: {len(self.samples)} samples from {data_path}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx: int):
        sample   = self.samples[idx]
        messages = [Message(**m) for m in sample['messages']]

        text   = format_chat(messages)
        tokens = self.tokenizer.encode(text)

        tokens = tokens[:self.max_length + 1]  # <1>

        labels = build_loss_mask(tokens, self.tokenizer)

        input_ids = tokens[:-1]   # <2>
        labels    = labels[1:]    # shift: labels[t] = tokens[t+1]

        length = len(input_ids)
        if self.pad_to_max:
            pad_len    = self.max_length - length
            input_ids  = input_ids  + [self.tokenizer.pad_id] * pad_len
            labels     = labels     + [-100]                  * pad_len

        return (
            torch.tensor(input_ids, dtype=torch.long),
            torch.tensor(labels,    dtype=torch.long),
        )


def collate_sft(batch):
    """
    Collate function for variable-length SFT samples.
    Pads to the longest sequence in the batch — avoids wasting compute
    padding all sequences to max_length when most are shorter.
    """
    input_ids, labels = zip(*batch)
    max_len = max(x.size(0) for x in input_ids)

    padded_inputs = torch.full((len(batch), max_len), 0,    dtype=torch.long)  # <3>
    padded_labels = torch.full((len(batch), max_len), -100, dtype=torch.long)

    for i, (x, y) in enumerate(zip(input_ids, labels)):
        padded_inputs[i, :x.size(0)] = x
        padded_labels[i, :y.size(0)] = y

    return padded_inputs, padded_labels

1. Truncate to `max_length + 1`: we need one extra token to create the shifted target.
2. Input is all tokens except the last; labels are all except the first — the standard language modeling shift.
3. Padding tokens get `input_id=0` (pad token) and `label=-100` (masked), so they contribute nothing to the loss.

### Generating a toy SFT dataset

For the nano model we construct a simple instruction dataset by treating Shakespeare passages as "creative writing" responses. Each sample pairs a generic writing prompt with an 80-word excerpt:

In [ ]:
def make_shakespeare_instruct(
    raw_text:    str,
    output_path: str,
    n_samples:   int = 500,
):
    """
    Build a toy SFT dataset where:
    - user asks to "continue the passage" or "write in the style of Shakespeare"
    - assistant responds with a Shakespeare excerpt
    """
    import random
    random.seed(42)

    prompts = [
        "Continue this passage in the style of Shakespeare:",
        "Write a short dramatic monologue.",
        "Write dialogue between two characters.",
        "Complete this Shakespearean verse:",
        "Write a soliloquy.",
    ]

    words  = raw_text.split()
    chunks = [' '.join(words[i:i+80]) for i in range(0, len(words)-80, 80)]

    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'w') as f:
        for i in range(min(n_samples, len(chunks))):
            sample = {
                "messages": [
                    {"role": "system",    "content": "You are a creative writing assistant."},
                    {"role": "user",      "content": random.choice(prompts)},
                    {"role": "assistant", "content": chunks[i]},
                ]
            }
            f.write(json.dumps(sample) + '\n')

    print(f"Wrote {min(n_samples, len(chunks))} samples to {output_path}")

## Full Fine-Tuning

Full fine-tuning trains all model parameters on the SFT dataset. It is simple and effective when the SFT dataset is large (> 100K samples), it fits in memory, and the distribution is not too far from pretraining. The training loop is nearly identical to pretraining — the only differences are the dataset class (`InstructDataset` instead of a token-stream dataset) and the loss function.

The masked cross-entropy loss used in SFT:

In [ ]:
def sft_loss(logits, labels):
    """
    Masked cross-entropy: only compute loss where labels != -100.
    logits: (B, T, vocab_size)
    labels: (B, T) with -100 at masked positions
    """
    B, T, V = logits.shape
    return F.cross_entropy(
        logits.view(B * T, V),
        labels.view(B * T),
        ignore_index=-100,
    )

**Catastrophic forgetting.** Full fine-tuning on a small SFT dataset can overwrite the pretrained weights to the point where the model loses general capabilities — it becomes good at responding to the specific prompts in the SFT data but loses the breadth learned during pretraining. The symptoms: eval loss on a held-out pretraining corpus increases during SFT, even while SFT eval loss decreases. Mitigations: use a [lower LR during SFT (typically 10× lower than the pretraining LR)]{.underline}, use fewer epochs (1–3 over the SFT data, not 10+), or use LoRA.

:::{.callout-caution}
## Catastrophic forgetting
Full fine-tuning on a small dataset (< 50K samples) routinely degrades general language modeling ability. Always evaluate perplexity on a held-out pretraining corpus during SFT to catch this early.

:::

## LoRA: Low-Rank Adaptation

### The core idea

A pretrained weight matrix $W_0 \in \mathbb{R}^{d \times k}$ is frozen. Fine-tuning is modeled as a [low-rank update]{.mark}:

$$W = W_0 + \Delta W = W_0 + BA$$

where $B \in \mathbb{R}^{d \times r}$ and $A \in \mathbb{R}^{r \times k}$ with rank $r \ll \min(d, k).$ During the forward pass the output becomes:

$$h = W_0 x + \Delta W x = W_0 x + BAx.$$

$W_0$ is frozen — it never receives gradients. Only $B$ and $A$ are trained. The number of trainable parameters per layer is:

$$|\theta_{\text{LoRA}}| = r(d + k) \quad \text{vs} \quad |\theta_{\text{full}}| = dk.$$

For a typical Transformer hidden dimension $d = k = 768$ and $r = 8$:

$$\frac{r(d + k)}{dk} = \frac{8 \times 1536}{768^2} = \frac{12288}{589824} \approx 2\%.$$

[LoRA uses approximately 2% of the parameters of full fine-tuning while achieving comparable task performance on most benchmarks.]{.mark}

### Initialization

$A$ is initialized with Kaiming uniform (standard for a linear layer). [$B$ is initialized to **zero**, so $\Delta W = BA = 0$ at the start of training]{.underline} — the LoRA model begins as an exact copy of the pretrained model. No warm-up is needed to prevent a sudden distribution shift at step 0.

:::{.callout-note}
Because $B = 0$ at initialization, the LoRA model's output is identical to the pretrained model's output before any gradient steps. This property makes it safe to resume fine-tuning from a checkpoint: the SFT loss at step 0 matches what you would see with the unmodified pretrained model.

:::

### The $\alpha / r$ scaling factor

The forward pass uses $\frac{\alpha}{r} BA x$ rather than $BAx$ directly. This scaling factor separates the [learning rate sensitivity from the choice of rank]{.underline}. When you change $r$, you can keep $\alpha$ fixed and the update magnitude stays the same. In practice: set $\alpha = r$ (scaling = 1) or $\alpha = 2r$ (scaling = 2). Values in $[1, 2]$ work well across model sizes.

We implement `LoRALinear` as a drop-in replacement for `nn.Linear`:

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


class LoRALinear(nn.Module):
    """
    A drop-in replacement for nn.Linear with LoRA adaptation.

    The forward pass computes:
        h = W_0 x + (alpha/r) * B * A * x

    W_0 is frozen. Only A and B are trained.
    """

    def __init__(
        self,
        in_features:  int,
        out_features: int,
        rank:         int   = 8,
        alpha:        float = None,
        dropout:      float = 0.0,
        bias:         bool  = True,
    ):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features
        self.rank         = rank
        self.alpha        = alpha if alpha is not None else float(rank)
        self.scaling      = self.alpha / self.rank

        self.weight = nn.Parameter(
            torch.empty(out_features, in_features), requires_grad=False  # <1>
        )
        self.bias_param = nn.Parameter(
            torch.zeros(out_features), requires_grad=False
        ) if bias else None

        self.lora_A = nn.Parameter(torch.empty(rank, in_features))       # <2>
        self.lora_B = nn.Parameter(torch.zeros(out_features, rank))      # <3>

        self.lora_dropout = nn.Dropout(p=dropout) if dropout > 0 else nn.Identity()

        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        # B is already zero — ensures delta_W = 0 at init

    @classmethod
    def from_linear(
        cls,
        linear:  nn.Linear,
        rank:    int   = 8,
        alpha:   float = None,
        dropout: float = 0.0,
    ) -> 'LoRALinear':
        has_bias = linear.bias is not None
        lora     = cls(
            linear.in_features, linear.out_features,
            rank=rank, alpha=alpha, dropout=dropout, bias=has_bias
        )
        lora.weight.data.copy_(linear.weight.data)          # <4>
        if has_bias:
            lora.bias_param.data.copy_(linear.bias.data)
        return lora

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        result   = F.linear(x, self.weight, self.bias_param)  # <5>
        lora_out = F.linear(
            self.lora_dropout(x),
            self.lora_B @ self.lora_A,
        ) * self.scaling
        return result + lora_out

    def merge(self) -> nn.Linear:
        """Fold LoRA into W_0; returns a plain nn.Linear with no LoRA overhead."""
        merged = nn.Linear(
            self.in_features, self.out_features,
            bias=self.bias_param is not None
        )
        merged.weight.data = self.weight.data + self.scaling * (self.lora_B @ self.lora_A)  # <6>
        if self.bias_param is not None:
            merged.bias.data = self.bias_param.data
        return merged

    def extra_repr(self) -> str:
        return (f'in={self.in_features}, out={self.out_features}, '
                f'rank={self.rank}, alpha={self.alpha:.1f}, '
                f'scaling={self.scaling:.3f}')

1. The pretrained weight is stored as a non-trainable `nn.Parameter` — it participates in `state_dict` serialization but receives no gradients.
2. $A \in \mathbb{R}^{r \times k}$: initialized with Kaiming uniform so gradients flow well from the first step.
3. $B \in \mathbb{R}^{d \times r}$: initialized to zero so $\Delta W = BA = 0$ at step 0.
4. `from_linear` copies the pretrained weights into the frozen `weight` and `bias_param` tensors.
5. The base path and the LoRA path are computed separately and summed. The base path is identical to a frozen `nn.Linear` call.
6. `merge` computes $W_{\text{merged}} = W_0 + \frac{\alpha}{r} BA$ and returns an ordinary `nn.Linear`.

## Injecting LoRA into the Model

The standard approach is to replace target `nn.Linear` layers in the model with `LoRALinear` layers in-place. The original LoRA paper applied LoRA to the query and value projection matrices in attention only. Subsequent empirical work shows that applying it to all linear layers typically performs better for instruction following.

In [ ]:
def inject_lora(
    model:        nn.Module,
    rank:         int       = 8,
    alpha:        float     = None,
    dropout:      float     = 0.0,
    target_names: list[str] = None,
) -> nn.Module:
    """
    Replace nn.Linear layers with LoRALinear layers in-place.

    target_names: list of substrings. A layer is replaced if its name
    contains any of the substrings. If None, all Linear layers are replaced.
    """
    if target_names is None:
        target_names = ['']   # matches everything

    replaced = 0
    for name, module in list(model.named_modules()):
        parts  = name.split('.')
        parent = model
        for part in parts[:-1]:
            parent = getattr(parent, part)
        attr = parts[-1]

        if (isinstance(module, nn.Linear) and
                any(t in name for t in target_names)):
            lora_layer = LoRALinear.from_linear(module, rank=rank,
                                                alpha=alpha, dropout=dropout)
            setattr(parent, attr, lora_layer)
            replaced += 1

    print(f"LoRA: replaced {replaced} Linear layers  (rank={rank}, alpha={alpha or rank})")
    return model


def freeze_base_model(model: nn.Module):
    """Freeze all parameters that are not LoRA matrices."""
    frozen = trained = 0
    for name, param in model.named_parameters():
        if 'lora_A' in name or 'lora_B' in name:
            param.requires_grad_(True)
            trained += param.numel()
        else:
            param.requires_grad_(False)
            frozen += param.numel()
    total = frozen + trained
    print(f"LoRA trainable: {trained:,} params  ({100*trained/total:.2f}% of {total/1e6:.1f}M)")


def count_trainable(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

### Verifying the initialization

After injection we verify two things: (1) LoRA matrices are the only trainable parameters, and (2) $\Delta W = 0$ at initialization, so the output of the LoRA model matches the pretrained model exactly:

In [ ]:
# Load pretrained model
model = GPT(config)
model.load_state_dict(torch.load('runs/nano_gpt/best_checkpoint.pt')['model'])

x     = torch.randint(0, config.vocab_size, (2, 32))
with torch.no_grad():
    out_before = model(x)[0].clone()

inject_lora(model, rank=8)
freeze_base_model(model)

with torch.no_grad():
    out_after = model(x)[0]

max_diff = (out_before - out_after).abs().max().item()
print(f"Max output difference after LoRA injection: {max_diff:.2e}")
assert max_diff < 1e-5, "LoRA injection changed model output — B init is wrong"
print("\u2713 LoRA injection correct: output unchanged at initialization")

## The LoRA Fine-Tuning Loop

The loop is nearly identical to pretraining, with three differences: a smaller learning rate (10× lower), fewer steps, and only LoRA parameters are updated by the optimizer. We load the pretrained model, inject LoRA, freeze the base weights, and run the standard train-eval cycle:

In [ ]:
from torch.utils.data import DataLoader

def sft_train(
    model_path:   str,
    data_path:    str,
    output_dir:   str,
    rank:         int   = 8,
    alpha:        float = None,
    lora_dropout: float = 0.05,
    max_lr:       float = 2e-4,    # 10× lower than pretraining
    min_lr:       float = 2e-5,
    warmup_steps: int   = 50,
    max_steps:    int   = 1000,
    batch_size:   int   = 4,
    max_length:   int   = 256,
    eval_every:   int   = 100,
):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    dtype  = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    # ---- Load pretrained model (GPT and Tokenizer from NB01, NB02) ----
    # GPT, NanoGPTConfig from NB01 (/courses/llm/01-gpt-architecture.html)
    # Tokenizer from NB02 (/courses/llm/02-tokenization.html)
    from nb01 import GPT, NanoGPTConfig
    from nb02 import Tokenizer

    config = NanoGPTConfig()
    model  = GPT(config).to(device)
    ckpt   = torch.load(model_path, map_location=device)
    model.load_state_dict(ckpt['model'] if 'model' in ckpt else ckpt)
    print(f"Loaded pretrained model from {model_path}")

    # ---- Inject LoRA ----
    inject_lora(model, rank=rank, alpha=alpha, dropout=lora_dropout)
    freeze_base_model(model)

    # ---- Data ----
    tok = Tokenizer.load('nano_tokenizer.json')

    train_ds = InstructDataset(data_path, tok, max_length=max_length)
    val_size  = max(1, len(train_ds) // 10)
    train_ds, val_ds = torch.utils.data.random_split(
        train_ds, [len(train_ds) - val_size, val_size]
    )

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        collate_fn=collate_sft, num_workers=2, pin_memory=True
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        collate_fn=collate_sft, num_workers=1
    )

    # ---- Optimizer — only LoRA parameters ----
    lora_params = [p for p in model.parameters() if p.requires_grad]
    optimizer   = torch.optim.AdamW(lora_params, lr=max_lr, weight_decay=0.1)

    # make_cosine_schedule from NB06 (/courses/llm/06-pretraining.html)
    from nb06 import make_cosine_schedule
    scheduler = make_cosine_schedule(
        optimizer,
        warmup_steps=warmup_steps,
        max_steps=max_steps,
        max_lr=max_lr,
        min_lr=min_lr,
    )

    # ---- Training loop ----
    model.train()
    step       = 0
    best_val   = float('inf')
    train_iter = iter(train_loader)

    while step < max_steps:
        # Refill iterator when exhausted
        try:
            x, y = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            x, y = next(train_iter)

        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        with torch.autocast(device_type=device.type, dtype=dtype,
                             enabled=torch.cuda.is_available()):
            logits, _ = model(x)
            loss      = sft_loss(logits, y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(lora_params, 1.0)
        optimizer.step()
        scheduler.step()
        step += 1

        if step % eval_every == 0 or step == max_steps:
            model.eval()
            val_losses = []
            with torch.no_grad():
                for vx, vy in val_loader:
                    vx, vy = vx.to(device), vy.to(device)
                    with torch.autocast(device_type=device.type, dtype=dtype,
                                        enabled=torch.cuda.is_available()):
                        vlogits, _ = model(vx)
                        val_losses.append(sft_loss(vlogits, vy).item())

            val_loss = sum(val_losses) / len(val_losses)
            lr_now   = scheduler.get_last_lr()[0]
            print(f"step {step:5d}  train_loss={loss.item():.4f}  "
                  f"val_loss={val_loss:.4f}  lr={lr_now:.2e}")

            if val_loss < best_val:
                best_val = val_loss
                lora_state = {
                    k: v for k, v in model.state_dict().items()
                    if 'lora_A' in k or 'lora_B' in k
                }
                torch.save(
                    {'lora_state': lora_state, 'rank': rank, 'alpha': alpha,
                     'step': step, 'val_loss': val_loss},
                    f"{output_dir}/best_lora.pt"
                )
            model.train()

    print(f"Training complete. Best val loss: {best_val:.4f}")
    return model

## Merging LoRA Weights

Before serving the model we can fold $\Delta W$ back into $W_0$:

$$W_{\text{merged}} = W_0 + \frac{\alpha}{r} BA.$$

[After merging, the model is a plain GPT with no LoRA overhead. Inference is identical to the pretrained model in speed and memory.]{.mark} The `merge_lora` function iterates over all `LoRALinear` layers, calls `.merge()` on each, and replaces it with the resulting `nn.Linear`:

In [ ]:
def merge_lora(model: nn.Module) -> nn.Module:
    """Replace all LoRALinear layers with merged nn.Linear layers."""
    merged_count = 0
    for name, module in list(model.named_modules()):
        if not isinstance(module, LoRALinear):
            continue
        parts  = name.split('.')
        parent = model
        for part in parts[:-1]:
            parent = getattr(parent, part)
        attr   = parts[-1]
        setattr(parent, attr, module.merge())
        merged_count += 1

    print(f"Merged {merged_count} LoRA layers into base weights")
    return model


def load_lora_checkpoint(
    model:     nn.Module,
    lora_path: str,
    rank:      int   = 8,
    alpha:     float = None,
):
    """
    Load a saved LoRA checkpoint into a model.
    Injects LoRA layers, then loads only the saved lora_A / lora_B weights.
    """
    ckpt = torch.load(lora_path, map_location='cpu')
    inject_lora(model, rank=ckpt.get('rank', rank), alpha=ckpt.get('alpha', alpha))
    missing, unexpected = model.load_state_dict(ckpt['lora_state'], strict=False)
    print(f"LoRA loaded: {len(ckpt['lora_state'])} tensors")
    if unexpected:
        print(f"  Unexpected keys: {unexpected}")
    return model

## What to Apply LoRA To

The original LoRA paper (Hu et al., 2021) applied LoRA to the query and value projections in attention only. Subsequent empirical work found:

| Target layers | Relative performance | Trainable params |
|---|---|---|
| Q, V only | Baseline | ~1% |
| Q, K, V, O | +2–5% on instruction tasks | ~2% |
| All attention + FFN | Best for instruction following | ~4% |
| All linear layers | Marginal improvement over above | ~6% |

: {tbl-colwidths="[40,35,25]"}

For the nano model, the distinction matters less — the model is small enough that full fine-tuning is feasible and the parameter budget is not a concern. For large models (7B+), the parameter efficiency of LoRA is the enabling factor: the difference between a run that fits in memory and one that does not.

**Rank selection.** $r = 8$ is a good default.[^lora_rank] The scaling factor $\alpha$ should be set equal to $r$ or $2r$.

[^lora_rank]: Use $r = 4$ for narrow task adaptation (e.g., style transfer on a small dataset). Use $r = 16$–$32$ when the fine-tuning distribution diverges significantly from pretraining (e.g., a new domain or language). There is rarely a reason to exceed $r = 64$ — at that scale, full fine-tuning with a lower LR is often simpler and equally effective.

:::{.callout-note}
For the nano 29.9M model, applying LoRA to all linear layers is perfectly fine — the total trainable parameter count is still well under 1M. For 7B+ models, the choice of target layers and rank has a real impact on GPU memory and training speed; all-attention + FFN is usually the best trade-off.

:::

## Summary

| Concept | Key detail |
|---|---|
| SFT loss masking | `ignore_index=-100` at prompt positions. Gradient only from response tokens. |
| Chat template | Consistent format between training and inference. Special role tokens in vocab. |
| Label shift | `labels[t]` = `tokens[t+1]`. Build mask on full sequence, then shift both. |
| Full fine-tuning risk | Catastrophic forgetting on small datasets. Use low LR (10×) and few epochs. |
| LoRA rank-decomposition | $\Delta W = BA$, $B \in \mathbb{R}^{d \times r}$, $A \in \mathbb{R}^{r \times k}$, $r \ll \min(d,k).$ |
| $B$ initialized to zero | $\Delta W = 0$ at init. Output identical to pretrained model at step 0. |
| $\alpha/r$ scaling | Controls LoRA update magnitude. Keep in $[1, 2].$ Default: $\alpha = r.$ |
| Trainable param fraction | $r(d+k)/(dk) \approx 2\%$ for $r=8$, $d=k=768.$ |
| Save only LoRA weights | `lora_A` + `lora_B` keys only — a fraction of full model size. |
| Merge before inference | $W_{\text{merged}} = W_0 + (\alpha/r)BA.$ Zero inference overhead after merge. |
| Best target layers | All attention + FFN linears for instruction following. Q+V only for narrow tasks. |

: {tbl-colwidths="[35,65]"}

## Exercises

**1.** Verify the loss mask implementation: create a short multi-turn conversation, tokenize it, build the loss mask, and print the decoded text for each position alongside its mask value. Confirm that all user and system tokens have `mask=-100` and all assistant tokens have `mask=token_id`.

**2.** Run the LoRA training loop for 500 steps with $r \in \{4, 8, 16\}.$ Plot eval loss vs. step for each rank. Verify that higher rank reaches lower eval loss but requires more steps to stabilize due to more parameters being optimized from zero.

**3.** Implement the forgetting check: before SFT, evaluate the pretrained model's perplexity on a held-out chunk of TinyShakespeare. After SFT (full fine-tuning, not LoRA), evaluate again and report the perplexity increase. Then repeat with LoRA — confirm that LoRA causes significantly less forgetting.

**4.** Implement `LoRALinear.merge()` independently and verify correctness: inject LoRA, train for 100 steps, then merge. The merged model's output on a test batch must match the unmerged LoRA model's output to within floating-point precision (`< 1e-5` max absolute difference).

**5.** Extend `inject_lora` to support per-layer rank: instead of a single `rank` argument, accept a `rank_map: dict[str, int]` where keys are layer name substrings and values are the rank to use for matching layers. Use this to assign `rank=16` to the first and last Transformer blocks (which tend to be most important for task adaptation) and `rank=4` to middle blocks.

**6.** Add a `lora_stats()` function that, given a model with LoRA layers, reports: total trainable params, total frozen params, trainable fraction, per-layer rank, and the current $\|\Delta W\|_F / \|W_0\|_F$ ratio for each LoRA layer. This ratio measures how much the LoRA adaptation has moved each layer from its pretrained initialization — a high ratio late in training may indicate overfitting to the SFT data.

■